# Phase 5s Wave 4 — Fidkowski-Kitaev gapped interface (Cayley calibration)

**Technical companion** to `lean/SKEFTHawking/FKGappedInterface.lean` and the bridge theorem `gapped_interface_dimensional_ladder` in `SPTClassification.lean`.

The Fidkowski-Kitaev (FK) interaction for 8 Majorana fermions is the unique Spin(7)-invariant quartic Hamiltonian on $\mathbb{R}^8$, built from the Cayley self-dual 4-form. It collapses the free-fermion $\mathbb{Z}$ classification of 1+1D BDI topological superconductors to $\mathbb{Z}_8 = \Omega_2^{\mathrm{Pin}^-}(\mathrm{pt})$. In our project it serves as the **2+1D rung of the dimensional-ladder evidence stack** for `gapped_interface_axiom` — the project's sole load-bearing axiom.

**Pipeline anchors:**
- Lean: 12 theorems / 0 sorry / native_decide on $\mathbb{Z}^{16\times 16}$ matrices
- Python: `src.core.formulas.fk_*` helpers (this notebook imports them)
- Bridge: `SKEFTHawking.SPTClassification.gapped_interface_dimensional_ladder`
- Source: Fidkowski & Kitaev, *Phys. Rev. B* **81**, 134509 (2010), Eq. 8
- Deep research: `Lit-Search/Phase-5s/Fidkowski-Kitaev interaction-...md`

**No inline physics is redefined here.** Every numerical claim is recomputed from the project's canonical helpers.

In [1]:
import numpy as np

from src.core.formulas import (
    fk_cayley_quartets,
    fk_dimensional_ladder_evidence,
    fk_eigenvalues,
    fk_ground_state_vector,
    fk_hamiltonian,
    fk_hamiltonian_sparse,
    fk_majorana_operators,
    fk_minimal_polynomial_residual,
    fk_parity_matrix,
    fk_spectral_gap,
)
from src.core.visualizations import (
    COLORS,
    fig_fk_dimensional_ladder,
    fig_fk_spectrum,
)

## 1. Eight Majorana operators via Jordan-Wigner

The 16-dimensional Fock space encodes 4 complex fermions $c_k = (\gamma_{2k-1} + i\gamma_{2k})/2$. Eight Hermitian Majoranas:

$$
\gamma_1 = \sigma_x \otimes I \otimes I \otimes I, \quad
\gamma_2 = \sigma_y \otimes I \otimes I \otimes I, \quad
\gamma_3 = \sigma_z \otimes \sigma_x \otimes I \otimes I, \;\ldots\;
$$

Each $\gamma_a$ is Hermitian, squares to identity, and obeys $\{\gamma_a, \gamma_b\} = 2\delta_{ab}$. We build them in code and verify the Clifford anticommutation as a sanity-check on the JW representation.

In [2]:
gammas = fk_majorana_operators()
I16 = np.eye(16, dtype=complex)

anticomm_max_dev = 0.0
for a in range(8):
    for b in range(8):
        anticomm = gammas[a] @ gammas[b] + gammas[b] @ gammas[a]
        expected = 2 * I16 if a == b else np.zeros_like(I16)
        anticomm_max_dev = max(anticomm_max_dev, float(np.max(np.abs(anticomm - expected))))

anticomm_max_dev

0.0

## 2. The 14 signed Cayley quartets

The Cayley calibration $\Omega$ is the self-dual 4-form on $\mathbb{R}^8$ preserved by $\mathrm{Spin}(7) \subset \mathrm{SO}(8)$. Its 14 nonzero components $\Omega_{abcd}$ pair into 7 Hodge-dual pairs $(abcd, a'b'c'd')$ where $\{a,b,c,d\} \sqcup \{a',b',c',d'\} = \{1,\ldots,8\}$, both with the same sign. The FK Hamiltonian is

$$
W = \sum_{a<b<c<d} \Omega_{abcd} \, \gamma_a \gamma_b \gamma_c \gamma_d.
$$

In [3]:
quartets = fk_cayley_quartets()
len(quartets), [(q[:4], q[4]) for q in quartets[:7]]

(14,
 [((1, 2, 3, 4), 1),
  ((5, 6, 7, 8), 1),
  ((1, 2, 5, 6), 1),
  ((3, 4, 7, 8), 1),
  ((1, 2, 7, 8), 1),
  ((3, 4, 5, 6), 1),
  ((1, 3, 5, 7), 1)])

## 3. Constructive vs sparse $W$ — exact agreement

Two independent encodings of $W$ ship in `formulas.py`:

1. `fk_hamiltonian()` builds $W$ from the JW $\gamma$ matrices and the 14 Cayley quartets — the *physics-derived* form.
2. `fk_hamiltonian_sparse()` returns the 10-nonzero-entry sparse matrix that Lean's `FK.W` definition stores literally.

These must be entry-wise equal: a structural cross-validation that the JW-expanded Cayley sum collapses (under Spin(7) projection) to exactly the sparse form Lean machine-checks.

In [4]:
W_full = fk_hamiltonian()
W_lean = fk_hamiltonian_sparse()

agree = bool(np.array_equal(W_full, W_lean))
nonzero_count = int((W_lean != 0).sum())

agree, nonzero_count

(True, 10)

The 10 nonzero entries are: $W_{0,0} = W_{15,15} = -6$, $W_{0,15} = W_{15,0} = +8$, and $W_{ii} = +2$ for $i \in \{3,5,6,9,10,12\}$ (the six 2-bit-popcount even-parity indices). All entries on odd-parity Fock states vanish — a direct consequence of the self-duality $*_8 \Omega = \Omega$, which forces $W\Gamma = W$ where $\Gamma$ is fermion parity.

## 4. Spin(7) decomposition and the {1, 8, 7} multiplicity system

Under $\mathrm{Spin}(7)$ stabilising $\Omega$, the 16-dim Fock space decomposes

$$
\Gamma = +1 \text{ (even parity, 8-dim)} \;\rightarrow\; \mathbf{1} \oplus \mathbf{7}, \qquad
\Gamma = -1 \text{ (odd parity, 8-dim)} \;\rightarrow\; \mathbf{8}
$$

with eigenvalues $-14, +2, 0$ on the three irreps. Combining the trace relations

$$
\mathrm{tr}\,W = 0, \qquad \mathrm{tr}\,W^2 = 14 \cdot 16 = 224, \qquad \dim = 16
$$

with the minimal polynomial $W^3 + 12W^2 - 28W = 0$ (roots $\{-14, 0, 2\}$) uniquely determines the multiplicities $(1, 8, 7)$. All four constraints are machine-checked in Lean by `native_decide` on integer matrix arithmetic.

In [5]:
spectrum = fk_eigenvalues()
trace_residual = int(sum(e * m for e, m in spectrum.items()))
frobenius_residual = int(sum(e ** 2 * m for e, m in spectrum.items()))
min_poly_resid_max = int(np.max(np.abs(fk_minimal_polynomial_residual(W_full))))

{
    'spectrum': spectrum,
    'trace': trace_residual,
    'frobenius': frobenius_residual,
    'min_poly_max_abs_entry': min_poly_resid_max,
    'dim_sum': sum(spectrum.values()),
    'spectral_gap': fk_spectral_gap(),
}

{'spectrum': {-14: 1, 0: 8, 2: 7},
 'trace': 0,
 'frobenius': 224,
 'min_poly_max_abs_entry': 0,
 'dim_sum': 16,
 'spectral_gap': 14}

## 5. Ground state — Spin(7) singlet, even parity, gap $\Delta = 14$

The unique ground state is the Spin(7) singlet
$|GS\rangle = (|0000\rangle - |1111\rangle)/\sqrt{2}$ at energy $E_0 = -14$. The first excited level $E_1 = 0$ is the entire odd-parity sector (8-fold degenerate, annihilated by $W$), giving spectral gap $\Delta = 14$. The integer-valued ground-state vector $v_{\mathrm{gs}} = e_0 - e_{15}$ satisfies $W v_{\mathrm{gs}} = -14\, v_{\mathrm{gs}}$ exactly — verified in Lean by `eigenvalue_ground` via `native_decide`.

In [6]:
v_gs = fk_ground_state_vector()
parity = fk_parity_matrix()

{
    'eigenvalue_check': bool(np.array_equal(W_full @ v_gs, -14 * v_gs)),
    'parity_eigenvalue': int(np.dot(v_gs, parity @ v_gs) // np.dot(v_gs, v_gs)),
    'gap': fk_spectral_gap(),
    'parity_commutator_max': int(np.max(np.abs(W_full @ parity - parity @ W_full))),
}

{'eigenvalue_check': True,
 'parity_eigenvalue': 1,
 'gap': 14,
 'parity_commutator_max': 0}

## 6. The spectrum, visualised

Three energy levels with multiplicities (1, 8, 7) and the Spin(7) decomposition annotation. The spectral gap $\Delta = 14$ is robust — far from the small-gap regime where 4-Majorana FK chain perturbations could close it.

In [7]:
# viz-ref: fig_fk_spectrum
fig_fk_spectrum()

## 7. Wave 4 bridge theorem — dimensional-ladder evidence

The bridge theorem `SKEFTHawking.SPTClassification.gapped_interface_dimensional_ladder` packages two independent dimensional analogs of the conjectured `gapped_interface_axiom`:

| Dimension | Status | Lean witness | Framework | Gap |
| --- | --- | --- | --- | --- |
| 1+1D | PROVED | `VillainHamiltonian.k3450_gappable` | K-matrix gappability of the 3450 model | — |
| 2+1D | PROVED | `SKEFTHawking.FK.fk_summary` | Cayley-calibration $\mathrm{Spin}(7)$ interaction | $\Delta = 14$ |
| 3+1D | AXIOMATIZED | `gapped_interface_axiom` | TPF conjecture — open at literature frontier | — |

The 1+1D and 2+1D witnesses are *independent* of each other (different model classes, different proof frameworks). Together they bracket the 3+1D conjecture on both sides; the axiom remains because direct construction in 3+1D would require a Hilbert space too large for `native_decide`.

In [8]:
fk_dimensional_ladder_evidence()

{'1+1D': {'status': 'PROVED',
  'witness': 'VillainHamiltonian.k3450_gappable',
  'framework': 'K-matrix gappability (3450 model)',
  'gap': None},
 '2+1D': {'status': 'PROVED',
  'witness': 'SKEFTHawking.FK.fk_summary',
  'framework': 'Cayley-calibration Spin(7) interaction (FK 2010)',
  'gap': 14},
 '3+1D': {'status': 'AXIOMATIZED',
  'witness': 'gapped_interface_axiom',
  'framework': 'TPF conjecture — open at literature frontier',
  'gap': None}}

In [9]:
# viz-ref: fig_fk_dimensional_ladder
fig_fk_dimensional_ladder()

## 8. Bordism context — $\mathbb{Z}_8$ vs $\mathbb{Z}_{16}$

The FK $\mathbb{Z}_8$ collapse classifies 1+1D BDI fermionic SPT phases (time-reversal $T^2 = +1$) via $\Omega_2^{\mathrm{Pin}^-}(\mathrm{pt}) = \mathbb{Z}_8$. The 3+1D analog is $\Omega_4^{\mathrm{Pin}^+}(\mathrm{pt}) = \mathbb{Z}_{16}$ (DIII class, $T^2 = (-1)^F$); the **Smith homomorphism** $\Omega_4^{\mathrm{Pin}^+} \twoheadrightarrow \Omega_2^{\mathrm{Pin}^-}$ multiplies by 2, matching the physical picture in which T-breaking domain walls on a 2+1D surface carry 1+1D BDI modes.

Our 2+1D FK Lean module provides one explicit machine-checked anchor in this ladder. The 3+1D `gapped_interface_axiom` remains the project's last load-bearing assumption — the Stage 14 QI register tracks it under `qi-axiom-elimination`.

## Summary

- **Cross-layer parity:** `fk_hamiltonian()` (constructive) and `fk_hamiltonian_sparse()` (Lean encoding) agree entry-wise; both are 16$\times$16 integer matrices with 10 nonzero entries.
- **Spectral content:** eigenvalues $\{-14, 0, +2\}$ with multiplicities $(1, 8, 7)$, gap $\Delta = 14$, ground state unique with even fermion parity.
- **Lean witness:** 12 theorems / 0 sorry / native_decide on $\mathbb{Z}^{16\times 16}$, plus the bridge theorem `gapped_interface_dimensional_ladder`.
- **Project role:** the 2+1D rung of the dimensional-ladder evidence stack for `gapped_interface_axiom` (the project's sole remaining axiom).